In [1]:
#Baseline revisited
# testing a raw ClemAgent playing the clembench games
#credits to Philipp, I just adapted some things a bit

In [2]:
import os

# Specify the game name here (this code can be adapted to any 2-player game)
GAME_NAME = "taboo"

# Local clone location of the clembench repository
CLEMBENCH_HOME = r"C:\Users\white\Desktop\agents_experiments\clembench_v3"

# Expose CLEMBENCH_HOME so the clem framework can find the games
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

In [3]:
# Clone the clembench repo (safe to re-run; git will warn if it already exists)
#!git clone https://github.com/clp-research/clembench $CLEMBENCH_HOME

# Install the requirements into the Python kernel
#%pip install -r $CLEMBENCH_HOME/requirements.txt

# Make tqdm usable in Jupyter notebooks
#%pip install --upgrade ipywidgets jupyter_client

In [4]:
#!pip install clemcore==3.5.0

In [5]:
#!clem --version
#!clem list games -s $GAME_NAME

In [6]:
from playpen.agents import ClemAgent, ClemObservation
from clemcore.backends import load_model


.--------------..--------------..--------------..--------------..--------------..--------------..--------------.
|   ______     ||   _____      ||      __      ||  ____  ____  ||   ______     ||  _________   || ____  _____  |
|  |_   __ \   ||  |_   _|     ||     /  \     || |_  _||_  _| ||  |_   __ \   || |_   ___  |  |||_   \|_   _| |
|    | |__) |  ||    | |       ||    / /\ \    ||   \ \  / /   ||    | |__) |  ||   | |_  \_|  ||  |   \ | |   |
|    |  ___/   ||    | |   _   ||   / ____ \   ||    \ \/ /    ||    |  ___/   ||   |  _|  _   ||  | |\ \| |   |
|   _| |_      ||   _| |__/ |  || _/ /    \ \_ ||    _|  |_    ||   _| |_      ||  _| |___/ |  || _| |_\   |_  |
|  |_____|     ||  |________|  |||____|  |____|||   |______|   ||  |_____|     || |_________|  |||_____|\____| |
'--------------''--------------''--------------''--------------''--------------''--------------''--------------'



In [7]:
class BaselineAgentPlayer(ClemAgent):
    """
    Simple example agent.

    It calls the "clp-chat" model with the current interaction history and
    uses the model's response as the next guess.
    """

    def __init__(self):
        super().__init__()
        self.model = load_model("clp-chat", gen_args=dict(temperature=0.0, max_tokens=None))

    def act(self, last: ClemObservation) -> str:
        # Use the full history (which usually already includes the last observation)
        _, _, response_text = self.model.generate_response(self.history)
        # Observe own response in the interaction history
        self.observe(dict(role="user", content=response_text))
        return response_text

In [8]:
#player initialization
player1 = BaselineAgentPlayer()
player2 = BaselineAgentPlayer()

2026-03-07 18:57:45,421 - clemcore.backends - INFO - Found registered model spec that unifies with {"model_name":"clp-chat"} -> {'model_name': 'clp-chat', 'backend': 'openai_compatible', 'lookup_source': 'C:\\Users\\white\\Desktop\\agents_experiments\\notebooks\\model_registry.json', 'model_id': 'Qwen/Qwen3-VL-30B-A3B-Instruct-FP8'}
2026-03-07 18:57:45,426 - clemcore.backends - INFO - Found registry entry for backend openai_compatible -> {'backend': 'openai_compatible', 'file_name': 'openai_compatible_api.py', 'file_path': 'C:\\Users\\white\\anaconda3\\envs\\playpen-env\\lib\\site-packages\\clemcore\\backends\\openai_compatible_api.py', 'lookup_source': 'packaged'}
2026-03-07 18:57:45,429 - clemcore.backends - INFO - Dynamically import backend openai_compatible
2026-03-07 18:57:46,853 - clemcore.backends - INFO - Successfully loaded clp-chat model
2026-03-07 18:57:46,856 - clemcore.backends - INFO - Loading models took: 0:00:01.427600
2026-03-07 18:57:46,865 - clemcore.backends - INFO 

In [9]:
from clemcore.clemgame import env, episode_results_folder_callbacks

# Create callbacks to record the interactions in a folder; here we name the folder after the models the agent uses
callbacks = episode_results_folder_callbacks(run_dir="clp-chat", result_dir_path="playpen-records", player_model_infos="BaselineAgentPlayer")

game_env = env(
    GAME_NAME,
    single_pass=True,
    callbacks=callbacks
)
game_env.reset()

2026-03-07 18:57:46,959 - clemcore.cli - INFO - Found '1' game matching the game_selector="taboo"
2026-03-07 18:57:46,961 - clemcore.cli - INFO - {
  "game_name": "taboo",
  "description": "Taboo game between two agents where one has to describe a word for the other to guess.",
  "main_game": "taboo",
  "players": 2,
  "image": "none",
  "languages": [
    "en"
  ],
  "benchmark": [
    "0.9",
    "1.0",
    "1.5",
    "2.0",
    "3.0"
  ],
  "regression": "small",
  "roles": [
    "Describer",
    "Guesser"
  ],
  "game_path": "C:\\Users\\white\\Desktop\\agents_experiments\\clembench_v3\\taboo"
}
2026-03-07 18:57:46,963 - clemcore.run - INFO - Loading game benchmark for taboo
2026-03-07 18:57:47,713 - clemcore.run - INFO - Loading game benchmark for taboo took: 0:00:00.749103
2026-03-07 18:57:47,720 - clemcore.run - INFO - Prepared instance queue for taboo using 3 experiments ['high_en', 'medium_en', 'low_en'] and 15 instances in total.
2026-03-07 18:57:47,722 - clemcore.run - INFO - 

In [10]:
print("possible agents:", game_env.possible_agents)
# In most cases, roles will be the content description of what player_0 and player_1 are
print("likely mapping:", game_env.unwrapped.game_master.game_spec["roles"])
agent_mapping = {"player_0": player1, "player_1": player2}

possible agents: ['player_0', 'player_1']
likely mapping: ['Describer', 'Guesser']


In [ ]:
num_episodes = 15

all_episodes_data = []

for episode in range(num_episodes):
    game_env.reset()
    player1.reset()  #keep in mind that if guesser is an agent, then IT should be reset
    player2.reset()

    context_response_pairs = []
    for agent_id in game_env.agent_iter():
        context, reward, termination, truncation, info = game_env.last()
        if termination or truncation:
            response = None # we step one more time to remove the agent from the env (final reward was observed in last)
        else:
            response = agent_mapping[agent_id](context)
        context_response_pairs.append((agent_id, context, response, reward))
        game_env.step(response)

    all_episodes_data.append(context_response_pairs)

    print(f"Episode {episode + 1}/{num_episodes} completed with {len(context_response_pairs)} steps")

    print(f"Episode took these {len(context_response_pairs)} steps:")
    print("-" * 20)
    for idx, (agent_id, context, response, reward) in enumerate(context_response_pairs):
        print(f"Step {idx} / Reward {reward:.2f}:")
        print(f"Agent({agent_id}) <- Context:", context)
        print(f"Agent({agent_id}) -> Response:", response)
        print("-" * 20)